# PY-05 | Καθαρισμός Δεδομένων ΕΛΣΤΑΤ με Pandas: Ανεργία 2021

Στο προηγούμενο μάθημα γνωρίσαμε τα βασικά εργαλεία του **Pandas**.

Σε αυτό το μάθημα θα τα εφαρμόσουμε σε **πραγματικά αρχεία Excel της ΕΛΣΤΑΤ** από την Απογραφή Πληθυσμού-Κατοικιών 2021.

Θα χρησιμοποιήσουμε δύο πίνακες για να δημιουργήσουμε ένα καθαρό dataset σε επίπεδο **Δήμου** και να κάνουμε μία μικρή διερεύνηση της ανεργίας.

> Ο βασικός στόχος του επεισοδίου είναι το **data cleaning**.  
> Δεν θα κάνουμε ακόμη χαρτογράφηση — αυτό θα έρθει αργότερα με GeoPandas.

## Μαθησιακοί στόχοι

Μετά το τέλος του μαθήματος θα μπορείς να:

- επιθεωρείς ένα πραγματικό Excel πριν το διαβάσεις,
- εντοπίζεις πού αρχίζουν τα πραγματικά δεδομένα,
- παρακάμπτεις τίτλους και πολυεπίπεδες επικεφαλίδες,
- δίνεις καθαρά ονόματα στις στήλες,
- φιλτράρεις ένα συγκεκριμένο γεωγραφικό επίπεδο,
- χειρίζεσαι γεωγραφικούς κωδικούς ως identifiers,
- ελέγχεις data types, missing values και duplicates,
- δημιουργείς αναλυτικούς δείκτες,
- ενώνεις δύο πίνακες με `merge()`,
- ελέγχεις αν το merge έγινε σωστά,
- εξάγεις ένα καθαρό dataset για μελλοντική ανάλυση.

## 1. Δεδομένα

**ΕΛΣΤΑΤ — Οικονομικά χαρακτηριστικά / 2021**

- [Γ04. Οικονομικά ενεργός και μη ενεργός πληθυσμός, απασχολούμενοι κατά τομέα οικονομικής δραστηριότητας, άνεργοι. Δημοτικές Κοινότητες](https://www.statistics.gr/el/statistics?p_p_id=documents_WAR_publicationsportlet_INSTANCE_VBZOni0vs5VJ&p_p_lifecycle=2&p_p_state=normal&p_p_mode=view&p_p_cacheability=cacheLevelPage&p_p_col_id=column-2&p_p_col_count=4&p_p_col_pos=2&_documents_WAR_publicationsportlet_INSTANCE_VBZOni0vs5VJ_javax.faces.resource=document&_documents_WAR_publicationsportlet_INSTANCE_VBZOni0vs5VJ_ln=downloadResources&_documents_WAR_publicationsportlet_INSTANCE_VBZOni0vs5VJ_documentID=532722&_documents_WAR_publicationsportlet_INSTANCE_VBZOni0vs5VJ_locale=el)

- [Β14. Άνεργοι κατά επίπεδο εκπαίδευσης. Δήμοι](https://www.statistics.gr/el/statistics?p_p_id=documents_WAR_publicationsportlet_INSTANCE_VBZOni0vs5VJ&p_p_lifecycle=2&p_p_state=normal&p_p_mode=view&p_p_cacheability=cacheLevelPage&p_p_col_id=column-2&p_p_col_count=4&p_p_col_pos=2&_documents_WAR_publicationsportlet_INSTANCE_VBZOni0vs5VJ_javax.faces.resource=document&_documents_WAR_publicationsportlet_INSTANCE_VBZOni0vs5VJ_ln=downloadResources&_documents_WAR_publicationsportlet_INSTANCE_VBZOni0vs5VJ_documentID=532726&_documents_WAR_publicationsportlet_INSTANCE_VBZOni0vs5VJ_locale=el)

Κατέβασε τα δύο αρχεία και τοποθέτησέ τα στον φάκελο:

```text
data/raw/
```

με τα αρχικά τους ονόματα:

```text
A1602_SAM04_TB_DC_00_2021_C04_F_GR.xlsx
A1602_SAM04_TB_DC_00_2021_B14_F_GR.xlsx
```

## 2. Imports και paths

Το notebook υποθέτει ότι χρησιμοποιείς το ενεργό **system Python** environment.

Αν το `read_excel()` ζητήσει engine για αρχεία `.xlsx`, μπορεί να χρειαστεί:

In [ ]:
%pip install openpyxl

In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
DATA_DIR = Path("data/raw")
OUTPUT_DIR = Path("data/processed")

c04_path = DATA_DIR / "A1602_SAM04_TB_DC_00_2021_C04_F_GR.xlsx"
b14_path = DATA_DIR / "A1602_SAM04_TB_DC_00_2021_B14_F_GR.xlsx"

print(c04_path)
print(b14_path)

## 3. Πριν καθαρίσουμε: επιθεώρηση των Excel

Σε πραγματικά δεδομένα δεν πρέπει να υποθέτουμε ότι η πρώτη γραμμή του Excel είναι και η πραγματική επικεφαλίδα του πίνακα.

Πρώτα βλέπουμε τα ονόματα των φύλλων.

In [ ]:
c04_excel = pd.ExcelFile(c04_path)
b14_excel = pd.ExcelFile(b14_path)

print("Γ04 sheets:", c04_excel.sheet_names)
print("Β14 sheets:", b14_excel.sheet_names)

Τώρα διαβάζουμε μόνο τις πρώτες γραμμές **χωρίς header**, ώστε να δούμε την πραγματική δομή.

In [ ]:
c04_preview = pd.read_excel(
    c04_path,
    sheet_name="Γ04",
    header=None,
    nrows=10
)

c04_preview

In [ ]:
b14_preview = pd.read_excel(
    b14_path,
    sheet_name="Β14",
    header=None,
    nrows=10
)

b14_preview

### Τι παρατηρούμε;

Τα Excel έχουν:

- τίτλο πίνακα,
- περιγραφή γεωγραφικού επιπέδου,
- κενές γραμμές,
- πολυεπίπεδες / nested επικεφαλίδες.

Άρα δεν είναι ιδανικό να αφήσουμε το Pandas να μαντέψει μόνο του τα column names.

Θα παρακάμψουμε τις επικεφαλίδες του Excel και θα δώσουμε **δικά μας καθαρά ονόματα**.

### Καλή πρακτική για τη διάθεση στατιστικών δεδομένων

Η ανάγκη για `header=None`, διαφορετικό `skiprows` και χειροκίνητα ονόματα στηλών δεν είναι πρόβλημα του Pandas. Προκύπτει επειδή αυτά τα Excel έχουν σχεδιαστεί κυρίως ως **πίνακες παρουσίασης για ανθρώπινη ανάγνωση** και όχι ως machine-readable datasets.

Μια καλύτερη πρακτική για έναν πάροχο επίσημων στατιστικών θα ήταν να διαθέτει παράλληλα:

- **tidy / machine-readable έκδοση** του πίνακα, όπου κάθε μεταβλητή είναι μία στήλη και κάθε παρατήρηση μία γραμμή,
- σταθερά και τεκμηριωμένα ονόματα μεταβλητών,
- CSV ή άλλο ανοικτό machine-readable format,
- **API ή query interface**, ώστε ο χρήστης να επιλέγει έτος, γεωγραφικό επίπεδο και μεταβλητές χωρίς να κατεβάζει πολλά διαφορετικά Excel,
- metadata με ορισμούς μεταβλητών, μονάδες μέτρησης, γεωγραφικό επίπεδο και έκδοση / αναθεώρηση του dataset.

Παραδείγματα τέτοιων πρακτικών υπάρχουν στο [ONS Developer Hub](https://developer.ons.gov.uk/), στο [ABS Data Explorer / Data API](https://www.abs.gov.au/statistics/application-programming-interfaces-apis/data-explorer-user-guide/generating-api-calls) και στα [Eurostat APIs](https://ec.europa.eu/eurostat/web/user-guides/data-browser/api-data-access/api-getting-started).

> **Σημαντικό:** αυτό δεν σημαίνει ότι το μορφοποιημένο Excel είναι άχρηστο. Είναι πολύ καλό για ανθρώπινη ανάγνωση. Ιδανικά όμως θα πρέπει να υπάρχει **και** ξεχωριστή έκδοση για προγραμματιστική ανάλυση.


# Μέρος Α — Καθαρισμός Γ04

## 4. Διαβάζουμε τα πραγματικά δεδομένα του Γ04

Στο Γ04 τα δεδομένα αρχίζουν μετά τις πρώτες 7 γραμμές του Excel.

Χρησιμοποιούμε:

- `skiprows=7`
- `header=None`

ώστε να διαβάσουμε μόνο τον πραγματικό πίνακα δεδομένων.

In [ ]:
c04 = pd.read_excel(
    c04_path,
    sheet_name="Γ04",
    skiprows=7,
    header=None
)

c04.head()

## 5. Καθαρά ονόματα στηλών

Αντί να διατηρήσουμε τα nested Excel headers, δημιουργούμε μία επίπεδη και αναγνώσιμη δομή.

In [ ]:
c04.columns = [
    "geo_level",
    "geo_code",
    "municipality",
    "population_total",
    "economically_active",
    "employed_total",
    "primary_sector",
    "secondary_sector",
    "tertiary_sector",
    "unemployed",
    "economically_inactive"
]

c04.head()

## 6. Επιθεώρηση της δομής

In [ ]:
print("Shape:", c04.shape)
print()
c04.info()

Ο πίνακας Γ04 περιλαμβάνει **πολλά γεωγραφικά επίπεδα** στο ίδιο αρχείο.

Ας δούμε πόσες εγγραφές υπάρχουν για κάθε `geo_level`.

In [ ]:
c04["geo_level"].value_counts().sort_index()

Στο συγκεκριμένο αρχείο:

- `0` → σύνολο χώρας
- `1` → μεγάλη γεωγραφική ενότητα
- `3` → Περιφέρεια
- `4` → Περιφερειακή Ενότητα
- `5` → Δήμος
- `6` → Δημοτική Ενότητα
- `7` → Δημοτική Κοινότητα

Για τη δική μας ανάλυση θέλουμε **μόνο Δήμους**.

## 7. Φιλτράρουμε μόνο τους Δήμους

Χρησιμοποιούμε `.copy()` επειδή θέλουμε να δημιουργήσουμε ένα ανεξάρτητο DataFrame που θα τροποποιήσουμε στη συνέχεια.

In [ ]:
c04_municipalities = c04[
    c04["geo_level"] == 5
].copy()

print("Municipalities:", len(c04_municipalities))

c04_municipalities.head()

## 8. Καθαρισμός του γεωγραφικού κωδικού

Ο γεωγραφικός κωδικός είναι **identifier**, όχι μέτρηση.

Δεν θέλουμε:

- να υπολογίσουμε μέσο όρο κωδικών,
- να τους αθροίσουμε,
- να εμφανίζονται ως δεκαδικοί αριθμοί.

Τους μετατρέπουμε επομένως σε string.

In [ ]:
c04_municipalities["geo_code"] = (
    pd.to_numeric(
        c04_municipalities["geo_code"],
        errors="coerce"
    )
    .astype("Int64")
    .astype("string")
)

c04_municipalities[["geo_code", "municipality"]].head()

### Καλή πρακτική: σταθεροί γεωγραφικοί κωδικοί

Οι γεωγραφικοί κωδικοί είναι το **κλειδί σύνδεσης** μεταξύ στατιστικών πινάκων και χωρικών δεδομένων.

Ιδανικά, ένας πάροχος δεδομένων θα πρέπει να διαθέτει:

- τον ίδιο σταθερό κωδικό στα στατιστικά δεδομένα και στα επίσημα όρια,
- σαφή τεκμηρίωση της κωδικοποίησης,
- πίνακες αντιστοίχισης (**concordance tables**) όταν αλλάζουν οι διοικητικές διαιρέσεις ή οι κωδικοί.

Έτσι ο αναλυτής δεν χρειάζεται να βασίζεται σε ονόματα Δήμων ή σε μη τεκμηριωμένους μετασχηματισμούς των κωδικών για να κάνει ένα spatial join.

Θα επιστρέψουμε σε αυτό το θέμα όταν συνδέσουμε τα δεδομένα με γεωμετρίες στο **GeoPandas**.


## 9. Έλεγχος αριθμητικών στηλών

Ορίζουμε ποιες στήλες πρέπει να είναι αριθμητικές.

In [ ]:
c04_numeric = [
    "population_total",
    "economically_active",
    "employed_total",
    "primary_sector",
    "secondary_sector",
    "tertiary_sector",
    "unemployed",
    "economically_inactive"
]

c04_municipalities[c04_numeric] = (
    c04_municipalities[c04_numeric]
    .apply(pd.to_numeric, errors="coerce")
)

c04_municipalities.dtypes

## 10. Missing values

Ακόμη κι αν πιστεύουμε ότι ένα dataset δεν έχει missing values, **πάντα το ελέγχουμε**.

In [ ]:
c04_municipalities.isna().sum()

Αν το αποτέλεσμα είναι `0` για όλες τις στήλες, δεν χρειάζεται να κάνουμε καμία επέμβαση.

> Δεν χρησιμοποιούμε `fillna(0)` χωρίς λόγο.  
> Missing value και πραγματική τιμή μηδέν είναι διαφορετικά πράγματα.

## 11. Duplicates

Σε ένα dataset με μία εγγραφή ανά Δήμο, ο γεωγραφικός κωδικός πρέπει να είναι μοναδικός.

In [ ]:
print(
    "Duplicate municipality codes:",
    c04_municipalities["geo_code"].duplicated().sum()
)

print(
    "Unique municipality codes:",
    c04_municipalities["geo_code"].nunique()
)

## 12. Δημιουργία δείκτη ανεργίας

Ο αριθμός των ανέργων από μόνος του δεν είναι κατάλληλος για σύγκριση Δήμων διαφορετικού μεγέθους.

Θέλουμε **ποσοστό ανεργίας**.

Στο συγκεκριμένο dataset χρησιμοποιούμε:

```text
unemployment_rate = unemployed / economically_active × 100
```

Το denominator είναι ο **οικονομικά ενεργός πληθυσμός**, όχι ο συνολικός πληθυσμός.

In [ ]:
c04_municipalities["unemployment_rate"] = (
    c04_municipalities["unemployed"]
    / c04_municipalities["economically_active"]
    * 100
)

c04_municipalities[
    [
        "municipality",
        "economically_active",
        "unemployed",
        "unemployment_rate"
    ]
].head()

In [ ]:
c04_municipalities["unemployment_rate"].describe()

# Μέρος Β — Καθαρισμός Β14

## 13. Διαβάζουμε το Β14

Στο Β14 τα πραγματικά δεδομένα αρχίζουν μετά τις πρώτες 5 γραμμές.

In [ ]:
b14 = pd.read_excel(
    b14_path,
    sheet_name="Β14",
    skiprows=5,
    header=None
)

b14.head()

## 14. Καθαρά ονόματα στηλών

Οι αρχικές επικεφαλίδες εκπαίδευσης είναι πολύ μεγάλες.

Για την ανάλυση δημιουργούμε σύντομα και σαφή ονόματα.

In [ ]:
b14.columns = [
    "geo_level",
    "geo_code",
    "municipality",
    "unemployed_total",
    "edu_tertiary",
    "edu_postsecondary",
    "edu_upper_secondary",
    "edu_vocational_lower_secondary",
    "edu_primary",
    "edu_low_or_no_schooling"
]

b14.head()

Οι κατηγορίες αντιστοιχούν, σε συντομευμένη μορφή, στις κατηγορίες του αρχικού πίνακα ΕΛΣΤΑΤ.

Δεν αλλάζουμε τις τιμές — μόνο τα ονόματα των στηλών ώστε ο κώδικας να είναι πιο ευανάγνωστος.

## 15. Κρατάμε μόνο τους Δήμους

In [ ]:
b14_municipalities = b14[
    b14["geo_level"] == 5
].copy()

print("Municipalities:", len(b14_municipalities))

## 16. Καθαρίζουμε τον γεωγραφικό κωδικό

In [ ]:
b14_municipalities["geo_code"] = (
    pd.to_numeric(
        b14_municipalities["geo_code"],
        errors="coerce"
    )
    .astype("Int64")
    .astype("string")
)

## 17. Αριθμητικές στήλες

In [ ]:
education_columns = [
    "edu_tertiary",
    "edu_postsecondary",
    "edu_upper_secondary",
    "edu_vocational_lower_secondary",
    "edu_primary",
    "edu_low_or_no_schooling"
]

b14_numeric = [
    "unemployed_total",
    *education_columns
]

b14_municipalities[b14_numeric] = (
    b14_municipalities[b14_numeric]
    .apply(pd.to_numeric, errors="coerce")
)

b14_municipalities.dtypes

## 18. Missing values και duplicates

In [ ]:
b14_municipalities.isna().sum()

In [ ]:
print(
    "Duplicate municipality codes:",
    b14_municipalities["geo_code"].duplicated().sum()
)

print(
    "Unique municipality codes:",
    b14_municipalities["geo_code"].nunique()
)

# Μέρος Γ — Validation πριν το merge

## 19. Αθροίζονται οι εκπαιδευτικές κατηγορίες ακριβώς στο σύνολο;

Αυτός είναι ένας πολύ χρήσιμος πραγματικός έλεγχος.

Δημιουργούμε το άθροισμα των εκπαιδευτικών κατηγοριών:

In [ ]:
b14_municipalities["education_sum"] = (
    b14_municipalities[education_columns]
    .sum(axis=1)
)

b14_municipalities["education_difference"] = (
    b14_municipalities["unemployed_total"]
    - b14_municipalities["education_sum"]
)

b14_municipalities[
    [
        "municipality",
        "unemployed_total",
        "education_sum",
        "education_difference"
    ]
].head(10)

Ας δούμε πόσοι Δήμοι έχουν ακριβώς μηδενική διαφορά.

In [ ]:
(
    b14_municipalities["education_difference"]
    .value_counts()
    .sort_index()
)

### Σημαντικό

Στο πραγματικό αρχείο μπορεί να παρατηρήσεις μικρές διαφορές ανάμεσα:

- στο δημοσιευμένο `unemployed_total`,
- και στο άθροισμα των επιμέρους εκπαιδευτικών κατηγοριών.

Δεν πρέπει να «διορθώσουμε» αυτές τις τιμές αυθαίρετα.

Το workbook που χρησιμοποιούμε δεν μας δίνει μέσα στον ίδιο τον πίνακα επαρκή πληροφορία για να εξηγήσουμε την αιτία της διαφοράς. Επομένως:

> **διατηρούμε τις επίσημες τιμές όπως δημοσιεύονται και καταγράφουμε τον έλεγχο.**

Αυτό είναι ουσιαστικό μέρος του data cleaning: **validation δεν σημαίνει ότι αλλάζουμε πάντα τα δεδομένα**.

## 20. Ποσοστιαία σύνθεση των ανέργων κατά εκπαίδευση

Τώρα μπορούμε να δημιουργήσουμε ποσοστά.

Για παράδειγμα:

```text
tertiary_unemployed_pct =
unemployed_with_tertiary_education / total_unemployed × 100
```

Προσοχή: αυτό **δεν είναι ποσοστό ανεργίας πτυχιούχων**.

Μας λέει ποιο ποσοστό **όλων των ανέργων** του Δήμου ανήκει σε αυτή την εκπαιδευτική κατηγορία.

In [ ]:
education_pct_columns = []

for column in education_columns:
    pct_column = f"{column}_pct"

    b14_municipalities[pct_column] = (
        b14_municipalities[column]
        / b14_municipalities["unemployed_total"]
        * 100
    )

    education_pct_columns.append(pct_column)

b14_municipalities[
    [
        "municipality",
        "unemployed_total",
        "edu_tertiary_pct",
        "edu_upper_secondary_pct",
        "edu_primary_pct"
    ]
].head()

Επειδή οι κατηγορίες του αρχικού πίνακα δεν αθροίζονται πάντα ακριβώς στο δημοσιευμένο σύνολο, τα ποσοστά τους επίσης μπορεί να μην αθροίζονται **ακριβώς** στο 100%.

Δεν τα επανακλιμακώνουμε τεχνητά.

# Μέρος Δ — Merge των δύο πινάκων

## 21. Πριν το merge: σύγκριση κωδικών

Θέλουμε να ξέρουμε αν οι δύο πίνακες περιέχουν τους ίδιους Δήμους.

In [ ]:
print(
    "Γ04 municipality codes:",
    c04_municipalities["geo_code"].nunique()
)

print(
    "Β14 municipality codes:",
    b14_municipalities["geo_code"].nunique()
)

codes_only_c04 = set(c04_municipalities["geo_code"]) - set(
    b14_municipalities["geo_code"]
)

codes_only_b14 = set(b14_municipalities["geo_code"]) - set(
    c04_municipalities["geo_code"]
)

print("Codes only in Γ04:", len(codes_only_c04))
print("Codes only in Β14:", len(codes_only_b14))

## 22. Επιλέγουμε τις στήλες του Β14 που χρειαζόμαστε

Δεν χρειάζεται να μεταφέρουμε δεύτερη φορά το όνομα Δήμου ή άλλα περιττά πεδία.

In [ ]:
b14_for_merge = b14_municipalities[
    [
        "geo_code",
        "unemployed_total",
        *education_columns,
        *education_pct_columns
    ]
].copy()

b14_for_merge.head()

## 23. Merge με τον γεωγραφικό κωδικό

Χρησιμοποιούμε το Γ04 ως βασικό dataset και κάνουμε `left` merge.

Το:

```python
validate="one_to_one"
```

ζητά από το Pandas να επιβεβαιώσει ότι κάθε κωδικός εμφανίζεται μία φορά σε κάθε πίνακα.

Το:

```python
indicator=True
```

δημιουργεί προσωρινά τη στήλη `_merge`, ώστε να ελέγξουμε την προέλευση κάθε εγγραφής.

In [ ]:
unemployment = c04_municipalities.merge(
    b14_for_merge,
    on="geo_code",
    how="left",
    validate="one_to_one",
    indicator=True
)

unemployment["_merge"].value_counts()

Αν όλες οι εγγραφές είναι `both`, όλοι οι Δήμοι του βασικού πίνακα βρήκαν αντίστοιχο record στο Β14.

## 24. Έλεγχος του αριθμού ανέργων μεταξύ των δύο πινάκων

Οι δύο πίνακες περιέχουν ανεξάρτητα ένα σύνολο ανέργων.

Αυτό μας δίνει μία εξαιρετική ευκαιρία για validation.

In [ ]:
unemployment["unemployed_match"] = (
    unemployment["unemployed"]
    == unemployment["unemployed_total"]
)

unemployment["unemployed_match"].value_counts()

Μπορούμε επίσης να εμφανίσουμε μόνο τυχόν ασυμφωνίες:

In [ ]:
unemployment.loc[
    ~unemployment["unemployed_match"],
    [
        "geo_code",
        "municipality",
        "unemployed",
        "unemployed_total"
    ]
]

Αν δεν εμφανίζεται καμία γραμμή, τα δημοσιευμένα σύνολα ανέργων συμφωνούν σε επίπεδο Δήμου μεταξύ των δύο αρχείων.

## 25. Καθαρίζουμε τις προσωρινές στήλες

Αφού ολοκληρώσαμε το validation, δεν χρειαζόμαστε:

- `_merge`
- `unemployed_total` ως δεύτερο αντίγραφο του ίδιου μεγέθους
- `unemployed_match`

Τα αφαιρούμε από το τελικό dataset.

In [ ]:
unemployment = unemployment.drop(
    columns=[
        "_merge",
        "unemployed_total",
        "unemployed_match"
    ]
)

unemployment.head()

# Μέρος Ε — Μικρό deep dive στην ανεργία

## 26. Count vs rate

Ένα σημαντικό στατιστικό μάθημα είναι ότι:

- **unemployed** → αριθμός ανέργων
- **unemployment_rate** → ποσοστό ανέργων στον οικονομικά ενεργό πληθυσμό

Οι δύο μεταβλητές απαντούν σε διαφορετικές ερωτήσεις.

### Δήμοι με τον μεγαλύτερο αριθμό ανέργων

In [ ]:
unemployment.nlargest(
    10,
    "unemployed"
)[
    [
        "municipality",
        "unemployed",
        "economically_active",
        "unemployment_rate"
    ]
]

### Δήμοι με το μεγαλύτερο ποσοστό ανεργίας

In [ ]:
unemployment.nlargest(
    10,
    "unemployment_rate"
)[
    [
        "municipality",
        "unemployed",
        "economically_active",
        "unemployment_rate"
    ]
]

Οι δύο λίστες δεν χρειάζεται να είναι ίδιες.

Ένας πολύ μεγάλος Δήμος μπορεί να έχει πολλούς ανέργους αλλά όχι απαραίτητα το υψηλότερο ποσοστό ανεργίας.

## 27. Περιγραφικά στατιστικά του ποσοστού ανεργίας

In [ ]:
unemployment["unemployment_rate"].describe()

## 28. Εκπαιδευτική σύνθεση της ανεργίας

Ας δούμε για κάθε Δήμο ποια από τις εκπαιδευτικές κατηγορίες έχει το μεγαλύτερο πλήθος ανέργων.

Πρώτα δημιουργούμε έναν dictionary που μεταφράζει τα σύντομα column names σε πιο αναγνώσιμα labels.

In [ ]:
education_labels = {
    "edu_tertiary": "Tertiary",
    "edu_postsecondary": "Post-secondary",
    "edu_upper_secondary": "Upper secondary",
    "edu_vocational_lower_secondary": "Vocational / lower secondary",
    "edu_primary": "Primary",
    "edu_low_or_no_schooling": "Low / no schooling"
}

dominant_column = (
    unemployment[education_columns]
    .idxmax(axis=1)
)

unemployment["dominant_unemployed_education"] = (
    dominant_column.map(education_labels)
)

unemployment[
    [
        "municipality",
        "unemployment_rate",
        "dominant_unemployed_education"
    ]
].head(10)

Πόσοι Δήμοι ανήκουν σε κάθε κυρίαρχη κατηγορία;

In [ ]:
unemployment[
    "dominant_unemployed_education"
].value_counts()

### Προσοχή στην ερμηνεία

Αυτό **δεν** μας λέει ποια εκπαιδευτική ομάδα έχει τον μεγαλύτερο κίνδυνο ανεργίας.

Για να υπολογίσουμε κάτι τέτοιο θα χρειαζόμασταν ως denominator τον συνολικό οικονομικά ενεργό πληθυσμό **κάθε εκπαιδευτικής ομάδας**.

Εδώ περιγράφουμε μόνο τη **σύνθεση των ήδη ανέργων**.

# Μέρος ΣΤ — Τελικός έλεγχος και export

## 29. Τελικός έλεγχος

In [ ]:
print("Rows:", len(unemployment))
print("Unique municipality codes:", unemployment["geo_code"].nunique())
print("Duplicate codes:", unemployment["geo_code"].duplicated().sum())
print("Total missing values:", unemployment.isna().sum().sum())

Μπορούμε επίσης να δούμε τα τελικά column names:

In [ ]:
unemployment.columns.tolist()

## 30. Επιλέγουμε τα πεδία που θέλουμε να αποθηκεύσουμε

Κρατάμε τόσο τις αρχικές χρήσιμες μεταβλητές όσο και τους δείκτες που δημιουργήσαμε.

In [ ]:
final_columns = [
    "geo_code",
    "municipality",
    "population_total",
    "economically_active",
    "employed_total",
    "unemployed",
    "unemployment_rate",
    "economically_inactive",
    *education_columns,
    *education_pct_columns,
    "dominant_unemployed_education"
]

unemployment_clean = unemployment[
    final_columns
].copy()

unemployment_clean.head()

## 31. Export σε CSV

Δημιουργούμε τον φάκελο `data/processed/` αν δεν υπάρχει ήδη.

In [ ]:
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

output_path = (
    OUTPUT_DIR
    / "elstat_unemployment_2021_municipalities.csv"
)

unemployment_clean.to_csv(
    output_path,
    index=False
)

print(f"Saved to: {output_path}")

Το αρχείο που δημιουργήσαμε είναι πλέον πολύ πιο κατάλληλο για:

- Exploratory Data Analysis,
- visualization,
- GeoPandas,
- spatial joins,
- clustering,
- PCA,
- spatial statistics,
- machine learning.

Αυτό είναι το βασικό νόημα ενός reproducible cleaning workflow:

```text
RAW ELSTAT EXCEL
        ↓
inspect
        ↓
select correct rows
        ↓
clean names and types
        ↓
validate
        ↓
create indicators
        ↓
merge
        ↓
validate again
        ↓
ANALYSIS-READY DATASET
```

### Από το manual download σε ένα reproducible data pipeline

Στο μάθημα δημιουργήσαμε μόνοι μας ένα καθαρό CSV από τα αρχικά Excel. Αυτό είναι χρήσιμο για να μάθουμε τη διαδικασία και να ελέγχουμε κάθε βήμα.

Σε μια πιο ώριμη υποδομή ανοικτών δεδομένων, το ίδιο workflow θα μπορούσε να ξεκινά απευθείας από ένα επίσημο API:

```text
dataset + year + geography + variables
                    ↓
          machine-readable response
                    ↓
             validation
                    ↓
                analysis
```

Ένα τέτοιο σύστημα δεν καταργεί το data cleaning — μειώνει όμως το **περιττό cleaning που οφείλεται στη μορφοποίηση των αρχείων** και κάνει την ανάλυση πιο εύκολα αναπαραγώγιμη.

Για reproducibility είναι επίσης χρήσιμο κάθε dataset να έχει σαφή **version / revision information**, ώστε ένα notebook να μπορεί να τεκμηριώνει ακριβώς ποια έκδοση των επίσημων δεδομένων χρησιμοποίησε.


# Ασκήσεις

## Άσκηση 1 — Πάνω από τον μέσο όρο

Υπολόγισε το μέσο `unemployment_rate` των Δήμων και εμφάνισε μόνο τους Δήμους που βρίσκονται πάνω από αυτόν.

In [ ]:
# Γράψε τη λύση σου εδώ

## Άσκηση 2 — Count vs rate

Βρες:

1. τους 5 Δήμους με τον μεγαλύτερο αριθμό ανέργων,
2. τους 5 Δήμους με το μεγαλύτερο ποσοστό ανεργίας.

Είναι οι δύο λίστες ίδιες;

In [ ]:
# Γράψε τη λύση σου εδώ

## Άσκηση 3 — Εκπαίδευση

Βρες τους 10 Δήμους με το μεγαλύτερο:

```text
edu_tertiary_pct
```

Θυμήσου: ο δείκτης αυτός περιγράφει το ποσοστό των ανέργων του Δήμου που ανήκουν στην τριτοβάθμια εκπαιδευτική κατηγορία.

In [ ]:
# Γράψε τη λύση σου εδώ

## Άσκηση 4 — Validation

Έλεγξε αν:

```text
unemployed == unemployed_total
```

για όλους τους Δήμους **πριν** αφαιρέσεις τη δεύτερη στήλη από το merged dataset.

Πόσες ασυμφωνίες υπάρχουν;

In [ ]:
# Γράψε τη λύση σου εδώ

# Λύσεις

## Λύση Άσκησης 1

In [ ]:
mean_rate = unemployment["unemployment_rate"].mean()

above_average = unemployment[
    unemployment["unemployment_rate"] > mean_rate
]

above_average[
    [
        "municipality",
        "unemployment_rate"
    ]
].sort_values(
    by="unemployment_rate",
    ascending=False
)

## Λύση Άσκησης 2

In [ ]:
top_count = unemployment.nlargest(
    5,
    "unemployed"
)[
    ["municipality", "unemployed"]
]

top_rate = unemployment.nlargest(
    5,
    "unemployment_rate"
)[
    ["municipality", "unemployment_rate"]
]

print("Top 5 by count:")
print(top_count)

print("\nTop 5 by rate:")
print(top_rate)

## Λύση Άσκησης 3

In [ ]:
unemployment.nlargest(
    10,
    "edu_tertiary_pct"
)[
    [
        "municipality",
        "edu_tertiary_pct",
        "unemployment_rate"
    ]
]

## Λύση Άσκησης 4

Η άσκηση αυτή πρέπει να εκτελεστεί στο στάδιο του merge, πριν αφαιρέσουμε τη στήλη `unemployed_total`.

In [ ]:
validation = c04_municipalities[
    ["geo_code", "municipality", "unemployed"]
].merge(
    b14_municipalities[
        ["geo_code", "unemployed_total"]
    ],
    on="geo_code",
    how="left",
    validate="one_to_one"
)

validation["match"] = (
    validation["unemployed"]
    == validation["unemployed_total"]
)

print("Mismatches:", (~validation["match"]).sum())

# Σύνοψη

Σε αυτό το μάθημα χρησιμοποιήσαμε πραγματικά δεδομένα ΕΛΣΤΑΤ και είδαμε:

- `pd.ExcelFile()`
- `pd.read_excel()`
- `header=None`
- `skiprows`
- καθαρά column names
- filtering γεωγραφικού επιπέδου
- `.copy()`
- καθαρισμό geographic codes
- `pd.to_numeric()`
- έλεγχο missing values
- έλεγχο duplicates
- δημιουργία unemployment rate
- δημιουργία ποσοστιαίων δεικτών
- data validation
- `merge()`
- `validate="one_to_one"`
- `indicator=True`
- count vs rate
- export με `to_csv()`

Το σημαντικότερο μάθημα είναι ότι **data cleaning δεν σημαίνει απλώς να αλλάζουμε format**.

Σημαίνει:

> να κατανοούμε τι αντιπροσωπεύει κάθε γραμμή και κάθε μεταβλητή, να ελέγχουμε τις υποθέσεις μας και να μπορούμε να εξηγήσουμε πώς δημιουργήθηκε το τελικό dataset.